# Training yolo11m — Fold 0
Expected runtime on Kaggle T4: ~7.5 hours per fold (≈2.5× slower than yolo11n)

In [ ]:
# !pip install ultralytics rich -q

In [ ]:
import sys
sys.path.insert(0, '/kaggle/working/rsna-lumbar-yolo')
from pathlib import Path
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('Memory:', torch.cuda.get_device_properties(0).total_memory // 1e9, 'GB')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
FOLD = 0
MODEL = 'yolo11m'
PROJECT_DIR = Path('runs')

In [ ]:
import yaml
hyp = yaml.safe_load(open('configs/hyp_base.yaml'))
ds = yaml.safe_load(open('configs/dataset.yaml'))
print("=== Hyperparameters ===")
for k, v in hyp.items(): print(f"  {k}: {v}")
print(f"\n=== Dataset: nc={ds['nc']} classes ===")

In [ ]:
from src.train.train_single import build_fold_dataset_yaml
fold_yaml = build_fold_dataset_yaml(
    fold_k=FOLD,
    splits_dir=Path('data/splits'),
    processed_root=Path('data/processed'),
    base_dataset_yaml=Path('configs/dataset.yaml'),
    output_dir=Path('data/fold_datasets'),
)
print('Fold dataset yaml:', fold_yaml)
print(yaml.safe_load(open(fold_yaml)))

In [ ]:
import time
from src.train.train_single import run_training, load_class_weights
class_weights = load_class_weights(Path('data/class_weights.json'))
start = time.time()
print(f"Starting training: {MODEL} fold {FOLD} on {DEVICE}")
results = run_training(
    model_name=MODEL, fold_k=FOLD, dataset_yaml=fold_yaml,
    hyp_yaml=Path('configs/hyp_base.yaml'), class_weights=class_weights,
    project_dir=PROJECT_DIR, device=DEVICE,
)
print(f"Training completed in {(time.time()-start)/60:.1f} minutes")

In [ ]:
from rich.console import Console; from rich.table import Table
console = Console()
table = Table(title=f"{MODEL} Fold {FOLD} Results")
table.add_column("Metric"); table.add_column("Value")
for k, v in results.items():
    table.add_row(str(k), str(v))
console.print(table)

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
results_csv = PROJECT_DIR / f"{MODEL}_fold{FOLD}" / "results.csv"
if results_csv.exists():
    df = pd.read_csv(results_csv)
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    df['train/box_loss'].plot(ax=axes[0,0], title='Box Loss')
    df['train/cls_loss'].plot(ax=axes[0,1], title='Cls Loss')
    df['train/dfl_loss'].plot(ax=axes[1,0], title='DFL Loss')
    df['metrics/mAP50(B)'].plot(ax=axes[1,1], title='mAP@50')
    for ax in axes.flat: ax.set_xlabel('Epoch'); ax.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()

In [ ]:
from src.train.train_single import run_validation
if 'error' not in results and results.get('best_weights_path'):
    val_results = run_validation(MODEL, FOLD, Path(results['best_weights_path']),
                                 fold_yaml, DEVICE)
    print("Validation results:", val_results)

## Next steps
Run cells 6-9 for folds 1-4, or compare results with 03_train_yolo11n.ipynb

In [ ]:
print("04_train_yolo11m.ipynb Complete ✓")